## 0. Windows CPU environment

Run in ordinary Windows PowerShell from your user account. Use an approved Python 3.12 installation, or the uv commands below. Test GitHub in that terminal using the same git ls-remote command. A successful check there and a failed check in Codex indicate an environment-specific limitation. If both fail, ask the network administrator about approved access.

```powershell
Set-Location "$env:LEAD_PROJECT_ROOT"
uv python install 3.12
uv venv --python 3.12 --seed .venv-preprocess
.\.venv-preprocess\Scripts\python.exe -m pip install ipykernel nbformat mne numpy scipy pandas scikit-learn PyYAML psutil
git ls-remote https://github.com/jeongwon8u8/Medley_skkai refs/pull/3/head
git clone https://github.com/jeongwon8u8/Medley_skkai repos/Medley_skkai
git -C repos/Medley_skkai fetch origin refs/pull/3/head
git -C repos/Medley_skkai checkout --detach d5bf678556ab9aeb0acf44ba2e7f0c5ad1c3b6bb
```

If Python 3.12 is already registered with the Windows launcher, replace the two uv commands with `py -3.12 -m venv .venv-preprocess`. In VSCode select `.venv-preprocess\Scripts\python.exe`. CPU performs indexing, filtering and export. The local stage needs neither PyTorch nor pretrained weights.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from importlib import metadata
import bisect
import csv
import hashlib
import itertools
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import uuid

import tempfile
import zipfile
ROOT = Path(os.environ.get("LEAD_DATA_ROOT", str(Path.home() / "Videos")))
PROJECT = Path(os.environ.get("LEAD_PROJECT_ROOT", str(ROOT / "LEAD_finetuning")))
ROOTS = {"turku": ROOT / "turku", "unm": ROOT / "ds003490", "ucsd": ROOT / "ds002778"}
CFG = dict(run="pd_full_seed0_v1", seed=0, strategy="full", last_layers=2,
           encoder_lr=1e-5, head_lr=1e-3, batch_size=32, accumulation=1, epochs=30,
           patience=5, weight_decay=0.01, clip=1.0, amp=True, threshold=0.5,
           num_workers=0, train_fraction=0.70, val_fraction=0.15, test_fraction=0.15,
           cpu_smoke=False, resume=False, prepare_full_cache=False,
           allow_recording_exclusions=False, overlap_acknowledgment="",
           unm_marker_guard_sec=1.0)
# Set explicit paths or dataset aliases here if the supplied files use different names.
INPUT_OVERRIDES = {}
COHORT_NAMES = {"turku": "turku", "turku_pd": "turku", "new mexico": "unm",
                "unm": "unm", "unm_pd": "unm", "ds003490": "unm",
                "ucsd": "ucsd", "ucsd_pd": "ucsd", "ds002778": "ucsd"}

CFG["amp"] = False
CFG["prepare_full_cache"] = True
CFG["turku_transfer_permission_reference"] = ""
MEDLEY = PROJECT / "repos/Medley_skkai"
MEDLEY_COMMIT = "d5bf678556ab9aeb0acf44ba2e7f0c5ad1c3b6bb"
LEAD_COMMIT = "dddee9c94f13b83d76f41ac9d452a74f84a76c5b"

if sys.version_info[:2] != (3, 12):
    raise RuntimeError("Select the dedicated Python 3.12 kernel.")

import mne
import numpy as np
import pandas as pd
import psutil
import yaml
from sklearn.model_selection import train_test_split
versions = {name: metadata.version(name) for name in
            ("mne", "numpy", "scipy", "pandas", "scikit-learn", "pyyaml", "psutil")}
for folder in ("configs", "data", "repos", "cache", "splits", "logs", "transfer"):
    (PROJECT / folder).mkdir(parents=True, exist_ok=True)
def atomic(path, writer):
    path = Path(path)
    temporary = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    try:
        writer(temporary)
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)

def write_json(path, value):
    atomic(path, lambda p: p.write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf-8"))

def write_table(path, table, separator="\t"):
    atomic(path, lambda p: table.to_csv(p, index=False, sep=separator))

def sha_file(path):
    with Path(path).open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()


report = dict(python=sys.version, executable=sys.executable, device="cpu",
              versions=versions, ram_bytes=psutil.virtual_memory().total,
              free_disk_bytes=shutil.disk_usage(ROOT).free)
write_json(PROJECT / "logs/environment_report_local_preprocessing.json", report)
print(report)


## 1. Pinned Medley preprocessing

Import only Medley's EEG loader and preprocessing configuration. No model is instantiated.

In [ ]:
def check_checkout(repo, commit):
    if not (repo / ".git").exists():
        raise FileNotFoundError(f"Clone the reference repository to {repo} and check out {commit}")
    def git(*args):
        return subprocess.check_output(["git", "-C", str(repo), *args], text=True).strip()
    if git("rev-parse", "HEAD") != commit or git("status", "--porcelain", "--untracked-files=no"):
        raise RuntimeError(f"Expected clean tracked source at {commit} in {repo}")


check_checkout(MEDLEY, MEDLEY_COMMIT)
MEDLEY_PY = MEDLEY / "medley_pd"
sys.path.insert(0, str(MEDLEY_PY))
from medley import eeg as medley_eeg
if not Path(medley_eeg.__file__).is_relative_to(MEDLEY_PY):
    raise RuntimeError("Restart the kernel to load the pinned Medley checkout.")
ENC = yaml.safe_load((MEDLEY_PY / "configs/encoders/lead.yaml").read_text(encoding="utf-8"))
CHANNELS = "Fp1 Fp2 F7 F3 Fz F4 F8 T7 C3 Cz C4 T8 P7 P3 Pz P4 P8 O1 O2".split()
assert ENC["channels"] == CHANNELS and ENC["pool"] == "channel"
assert (ENC["fs"], ENC["seg_sec"], ENC["amp_div"], ENC["montage"]) == (200, 2, 1.0, "standard_1005")
assert ENC["preprocess"] == {"l_freq": 0.5, "h_freq": 45.0, "rereference": "average"}

PP = dict(flag_uv=100.0, min_good_channels=20, notch=None, **ENC["preprocess"])


## 2. Team configurations and pool

Use the extracted team YAMLs under configs and the original private pool TSV. Preserve leading zeros and validate 121 eligible subjects.

In [ ]:
SCHEMA = """dataset subject_uid source_subject_id country label_pd age sex education_years
source_version original_id medication_ses_01 medication_ses_02 medication_observed
eligible_encoder_training exclusion_reason person_uid identity_link_status""".split()
INPUT_NAMES = ("turku_pd.yaml", "unm_pd.yaml", "ucsd_pd.yaml", "pd_hc.yaml",
               "pool_base_3cohorts_private.tsv")

def locate_inputs():
    found, problems = {}, []
    for name in INPUT_NAMES:
        paths = ([Path(INPUT_OVERRIDES[name])] if name in INPUT_OVERRIDES
                 else [PROJECT / "configs" / name] if name.endswith(".yaml") else [ROOT / name])
        if len(paths) != 1 or not paths[0].is_file():
            problems.append(f"{name}: {paths or 'missing'}")
        else:
            found[name] = paths[0]
    if problems:
        raise FileNotFoundError("Resolve input files in INPUT_OVERRIDES:\n" + "\n".join(problems))
    return found

def tsv(path):
    return pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False, encoding="utf-8-sig")

inputs = locate_inputs()
cohort_configs = {name: yaml.safe_load(inputs[name].read_text(encoding="utf-8-sig"))
                  for name in INPUT_NAMES if name.endswith(".yaml")}
if not all(isinstance(config, dict) and config for config in cohort_configs.values()):
    raise ValueError("Expected four nonempty YAML mappings.")
DATASET_CFG = {key: cohort_configs[f"{key}_pd.yaml"] for key in ROOTS}
CONDITIONS = {}
expected_conditions = {"turku": {"EO", "EC"}, "unm": {"EO", "EC"}, "ucsd": {"EO"}}
expected_lines = {"turku": 50, "unm": 60, "ucsd": 60}
for key, definition in DATASET_CFG.items():
    cond = definition["condition"]
    CONDITIONS[key] = [cond] if isinstance(cond, str) else list(cond)
    if (set(CONDITIONS[key]) != expected_conditions[key]
            or len(CONDITIONS[key]) != len(expected_conditions[key])
            or definition["name"] != f"{key}_pd"
            or definition["dir"] != ROOTS[key].name
            or definition["line_freq"] != expected_lines[key]
            or definition["file_ext"] != ["bdf" if key == "ucsd" else "set"]
            or not isinstance(definition["ref_aliases"], list)
            or definition["participants"] != "pd_hc_participants.tsv"
            or definition["columns"] != {"subject": "subject_uid", "group": "label_pd", "label_pd": "label_pd"}
            or definition["patient_regex"] != "^1$"):
        raise ValueError(f"Unexpected dataset YAML for {key}: review the original mapping.")
TASK_CFG = cohort_configs["pd_hc.yaml"]
if (TASK_CFG["type"], TASK_CFG["target"], TASK_CFG["positive"], TASK_CFG["patients_only"]) != (
        "classification", "label_pd", {"above": 0.5}, False):
    raise ValueError("The task YAML must specify binary PD/HC classification with both groups.")
pool_original = tsv(inputs[INPUT_NAMES[-1]])
if missing_columns := set(SCHEMA) - set(pool_original):
    raise ValueError(f"Pool columns missing: {sorted(missing_columns)}")
eligible = pool_original.eligible_encoder_training.str.strip().str.casefold().map(
    {"true": True, "yes": True, "1": True, "false": False, "no": False, "0": False})
if eligible.isna().any():
    raise ValueError("Unknown eligible_encoder_training value.")
pool = pool_original.loc[eligible].copy()
pool["cohort"] = pool.dataset.str.strip().str.casefold().map(COHORT_NAMES)
if pool.cohort.isna().any() or not pool.label_pd.isin(["0", "1"]).all():
    raise ValueError("Only the three specified cohorts and binary labels may be eligible.")
pool["label_pd"] = pool.label_pd.astype(int)
for field in ("subject_uid", "source_subject_id", "person_uid"):
    if pool[field].str.strip().eq("").any() or not pool[field].str.strip().eq(pool[field]).all():
        raise ValueError(f"Missing or untrimmed values in {field}.")
if pool.subject_uid.duplicated().any() or pool.duplicated(["cohort", "source_subject_id"]).any():
    raise ValueError("Nonunique subject or source identifier.")
if pool.groupby("person_uid").label_pd.nunique().gt(1).any():
    raise ValueError("A person has conflicting diagnoses.")
expected = {("turku", 0): 20, ("turku", 1): 20, ("unm", 0): 25, ("unm", 1): 25,
            ("ucsd", 0): 16, ("ucsd", 1): 15}
counts = pool.groupby(["cohort", "label_pd"]).size().to_dict()
if counts != expected:
    raise ValueError(f"Unexpected eligible counts: {counts}")
for cohort, group in pool.groupby("cohort"):
    folder = PROJECT / "data/generated_participant_tables" / cohort
    folder.mkdir(parents=True, exist_ok=True)
    write_table(folder / DATASET_CFG[cohort]["participants"], group[["subject_uid", "label_pd"]])
audit = {"identity_link_status": pool.identity_link_status.value_counts(dropna=False).to_dict(),
         "distinct_person_ids": int(pool.person_uid.nunique()),
         "eligible_counts": {f"{c}/{y}": int(n) for (c, y), n in counts.items()},
         "pool_ineligible": pool_original.loc[~eligible, ["subject_uid", "exclusion_reason"]].to_dict("records"),
         "input_sha256": {name: sha_file(path) for name, path in inputs.items()}}
print(audit["eligible_counts"])

## 3. EEG intervals and New Mexico marker review

Reuse the local file and event selection. For the recording with a short initial EC run, the initial EC run contains only five alternate markers from 0.374 to 4.374 s. Retain only 1.374–3.374 s after edge guards. For the recording with an isolated pulse, discard the isolated EC marker at 202.300 s and retain the later cadence-confirmed block. A changed or unreviewed anomaly stops indexing.

In [ ]:
EO_TYPE = "Eyes Open: Every 1000 ms"
EC_TYPE = "Eyes Closed: Every 1000 ms"
REST_MARKERS = {EO_TYPE: ("EO", {"S 1", "S 2"}), EC_TYPE: ("EC", {"S 3", "S 4"})}
NONREST = {("STATUS", "boundary"), ("STATUS", "S 85"), ("STATUS", "S255"),
           ("Standard Tone", "S201"), ("Target Tone", "S200"), ("Novel Tone", "S202")}

def unm_intervals(path, duration, guard):
    with path.open(encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle, delimiter="\t")
        if not {"onset", "duration", "trial_type", "value"}.issubset(reader.fieldnames or []):
            raise ValueError("Missing resting event columns.")
        events = list(reader)
    blocks, block, previous, notes = [], [], -math.inf, []
    if guard < 0:
        raise ValueError("Marker guard must be nonnegative.")
    for event in events:
        onset = float(event["onset"])
        kind, value = event["trial_type"], " ".join(event["value"].split())
        if not math.isfinite(onset) or onset < 0 or onset >= duration or onset < previous:
            raise ValueError(f"Invalid or unsorted event onset {onset} for duration {duration}.")
        previous = onset
        if kind not in REST_MARKERS:
            if (kind, value) not in NONREST:
                raise ValueError(f"Unrecognized event {kind!r} / {value!r}. Review local events before indexing.")
            if block:
                blocks.append(block)
                block = []
            continue
        condition, codes = REST_MARKERS[kind]
        if value not in codes or event["duration"] not in {"n/a", "", "0", "0.0"}:
            raise ValueError(f"Unexpected resting trigger or duration: {event}")
        if block:
            last_time, last_condition, last_code = block[-1]
            if condition != last_condition or not 0.95 <= onset - last_time <= 1.05 or value == last_code:
                blocks.append(block)
                block = []
        block.append((onset, condition, value))
    if block:
        blocks.append(block)
    intervals = []
    for block in blocks:
        start, stop = block[0][0] + guard, block[-1][0] - guard
        condition = block[0][1]
        if len(block) != 60:
            notes.append(f"{condition}: {len(block)} contiguous markers at {block[0][0]:.3f}s.")
        if stop - start >= 2 - 1e-8:
            intervals.append((condition, start, stop))
    if {part[0] for part in intervals} != {"EO", "EC"}:
        raise ValueError(f"Unable to establish both stable EO and EC intervals. Marker audit: {notes}")
    return intervals, notes

def review_unm_marker_runs(path, parts, notes):
    expected = {
        "EC: 5 contiguous markers at 0.374s.": (
            ["EC: 5 contiguous markers at 0.374s."], (1.374, 3.374),
            "EC retained only within the five alternating 1 Hz markers. "
            "After both guards this provides one 2-second window. The true start is unknown."),
        "EC: 1 contiguous markers at 202.300s.": (
            ["EC: 1 contiguous markers at 202.300s."], (402.742, 459.750),
            "The isolated 202.300 s EC pulse cannot define an interval. "
            "Only the later contiguous EC block is retained."),
    }
    if not notes:
        return None
    if notes[0] not in expected:
        raise ValueError(f"Unreviewed irregular marker run: {notes}")
    reviewed, endpoints, decision = expected[notes[0]]
    ec = [(a, b) for c, a, b in parts if c == "EC"]
    if notes != reviewed or len(ec) != 1 or any(abs(a-b) > 0.002 for a,b in zip(ec[0], endpoints)):
        raise ValueError(f"Reviewed marker run has changed: {path.name}, {notes}, {ec}")
    return decision

def medication(value):
    text = re.sub(r"[\s_-]+", " ", value.strip().casefold())
    if text in {"off", "off medication", "off med"}:
        return "off"
    if text in {"on", "on medication", "on med"}:
        return "on"
    if text in {"", "na", "n/a", "none", "no s2", "hc", "control", "not applicable",
                "not observed", "not recorded", "unknown"}:
        return None
    raise ValueError(f"Unrecognized medication state {value!r}")

def is_raw(path, root):
    return all("derivative" not in part.casefold() and "prep" not in part.casefold()
               for part in path.relative_to(root).parts)

def bids_id(source):
    return source if source.startswith("sub-") else f"sub-{source}"

def inspect_header(path):
    reader = mne.io.read_raw_eeglab if path.suffix.lower() == ".set" else mne.io.read_raw_bdf
    with reader(str(path), preload=False, verbose="ERROR") as raw:
        return (raw.n_times / raw.info["sfreq"], float(raw.info["sfreq"]),
                list(raw.ch_names), dict(Counter(raw.get_channel_types())))

for root in ROOTS.values():
    if not root.is_dir():
        raise FileNotFoundError(f"Set the explicit dataset root: {root}")
for cohort in ("unm", "ucsd"):
    source = DATASET_CFG[cohort]["openneuro"]
    accession = f"{source['id']}.v{source['tag']}"
    description = json.loads((ROOTS[cohort] / "dataset_description.json").read_text(encoding="utf-8-sig"))
    if not str(description.get("DatasetDOI", "")).endswith(accession):
        raise ValueError(f"{cohort}: unexpected dataset version, expected {accession}")
original_unm = tsv(ROOTS["unm"] / "participants.tsv").set_index("participant_id", verify_integrity=True)
turku_files = defaultdict(list)
for path in sorted(ROOTS["turku"].rglob("*.set")):
    if is_raw(path, ROOTS["turku"]) and (match := re.fullmatch(r"(.+)_eyes(Open|Closed)\.set", path.name)):
        turku_files[match[1], {"Open": "EO", "Closed": "EC"}[match[2]]].append(path)
audit["turku_raw_inventory"] = dict(Counter(condition for (_, condition), paths in turku_files.items()
                                             for path in paths))
audit.update(missing_files=[], invalid_intervals=[], interval_notes=[], recording_headers=[])
records = []
for row in pool.itertuples(index=False):
    c, sid, label = row.cohort, row.source_subject_id, row.label_pd
    root = ROOTS[c]
    try:
        selections = []
        if c == "turku":
            for condition in CONDITIONS[c]:
                paths = turku_files.get((sid, condition), [])
                if len(paths) > 1:
                    raise ValueError(f"Ambiguous raw {condition} recordings: {paths}")
                if paths:
                    state = row.medication_observed.strip().casefold() or "unknown"
                    selections.append((paths[0], "", condition, state if label else "hc"))
                else:
                    audit["missing_files"].append(dict(subject_uid=row.subject_uid, condition=condition))
        elif c == "unm":
            source = bids_id(sid)
            original = original_unm.loc[source]
            if original["Group"].strip().upper() != ("PD" if label else "CTL"):
                raise ValueError(f"BIDS diagnosis disagrees with pool for {source}.")
            states = [medication(original[f"sess{i}_Med"]) for i in (1, 2)]
            pool_states = [medication(row.medication_ses_01), medication(row.medication_ses_02)]
            if any(p is not None and p != s for p, s in zip(pool_states, states)):
                raise ValueError(f"Pool/BIDS medication disagreement: {pool_states} vs {states}.")
            paths = sorted((root / source).glob("ses-*/eeg/*_task-Rest_eeg.set"))
            if label:
                if sorted(s for s in states if s) != ["off", "on"]:
                    raise ValueError(f"Expected one ON and one OFF session, got {states}.")
                selected_session = f"ses-{states.index('off')+1:02d}"
                paths = [p for p in paths if p.parent.parent.name == selected_session]
            if len(paths) != 1:
                raise FileNotFoundError(f"Expected unique selected UNM recording for {source}, found {paths}.")
            selections.append((paths[0], paths[0].parent.parent.name, None, "off" if label else "hc"))
        else:
            source, session = bids_id(sid), ("ses-off" if label else "ses-hc")
            paths = sorted((root / source / session / "eeg").glob("*_task-rest_eeg.bdf"))
            if len(paths) != 1:
                raise FileNotFoundError(f"Expected unique UCSD recording for {source}/{session}, found {paths}.")
            selections.append((paths[0], session, CONDITIONS[c][0], "off" if label else "hc"))
    except (FileNotFoundError, ValueError, KeyError) as exc:
        audit["missing_files"].append(dict(subject_uid=row.subject_uid, reason=str(exc)))
        continue
    for path, session, condition, state in selections:
        try:
            if c == "unm":
                fdt = path.with_suffix(".fdt")
                if not fdt.is_file() or fdt.stat().st_size == 0:
                    raise FileNotFoundError(f"Missing or empty UNM recording binary: {fdt}")
            duration, fs, channel_names, channel_types = inspect_header(path)
            if fs != (512 if c == "ucsd" else 500):
                raise ValueError(f"Unexpected sampling rate {fs}.")
            if c == "ucsd":
                scalp = {name.casefold() for name in channel_names
                         if not name.casefold().startswith("exg") and name.casefold() != "status"}
                if len(scalp) != 32 or not set(x.casefold() for x in CHANNELS).issubset(scalp):
                    raise ValueError("Unexpected UCSD scalp channel inventory.")
            eeg_json = path.with_suffix(".json")
            sidecar = json.loads(eeg_json.read_text(encoding="utf-8-sig")) if eeg_json.is_file() else {}
            audit["recording_headers"].append(dict(eeg_path=str(path), fs=fs, duration=duration,
                channel_names=channel_names, channel_types=channel_types, reference=sidecar.get("EEGReference"),
                header_line_freq=sidecar.get("PowerLineFrequency")))
            if c == "unm":
                # The audited UNM event files are recording/session-specific.
                # A dataset-level description cannot substitute for the onset table.
                events = path.with_name(path.name.removesuffix("_eeg.set") + "_events.tsv")
                parts, notes = unm_intervals(events, duration, CFG["unm_marker_guard_sec"])
                decision = review_unm_marker_runs(events, parts, notes)
                if decision:
                    audit["interval_notes"].append(dict(eeg_path=str(path), notes=notes, decision=decision))
            else:
                parts = [(condition, 0.0, duration)]
            for cond, start, stop in parts:
                if cond not in CONDITIONS[c] or not (0 <= start < stop <= duration and stop - start >= 2 - 1e-8):
                    raise ValueError(f"Invalid {cond} interval {start}–{stop} for {duration}s recording.")
            for cond, start, stop in parts:
                records.append(dict(dataset=c, subject_uid=row.subject_uid, person_uid=row.person_uid,
                                    source_subject_id=sid, label_pd=label, eeg_path=str(path),
                                    session=session, condition=cond, medication_state=state,
                                    start_sec=start, stop_sec=stop))
        except Exception as exc:
            audit["invalid_intervals"].append(dict(subject_uid=row.subject_uid, eeg_path=str(path), reason=str(exc)))

INTERVAL_PATH = PROJECT / "data/eeg_intervals_private.tsv"
intervals = pd.DataFrame(records, columns="""dataset subject_uid person_uid source_subject_id label_pd eeg_path
                                           session condition medication_state start_sec stop_sec""".split())
if intervals.duplicated(["eeg_path", "start_sec", "stop_sec"]).any():
    raise ValueError("Duplicate intervals, potentially multiple identities for one file.")
for _, group in intervals.groupby("eeg_path"):
    ordered = group.sort_values("start_sec")
    if (ordered.start_sec.to_numpy()[1:] < ordered.stop_sec.to_numpy()[:-1]).any():
        raise ValueError("Overlapping intervals in one recording.")
audit["excluded_subjects_before_preprocessing"] = sorted(set(pool.subject_uid) - set(intervals.subject_uid))
audit["selected"] = [
    dict(dataset=c, subjects=int(g.subject_uid.nunique()), recordings=int(g.eeg_path.nunique()),
         subjects_by_label=g.drop_duplicates("subject_uid").label_pd.value_counts().to_dict(),
         intervals_by_condition=g.condition.value_counts().to_dict(),
         recordings_by_condition=g.groupby("condition").eeg_path.nunique().to_dict(),
         medication_sessions=g.drop_duplicates("eeg_path").groupby(["session", "medication_state"]).size()
             .rename("count").reset_index().to_dict("records"))
    for c, g in intervals.groupby("dataset")]
write_table(INTERVAL_PATH, intervals)
write_json(PROJECT / "logs/dataset_audit.json", audit)
print(audit["selected"])
unexpected_missing = [x for x in audit["missing_files"] if x.get("condition") != "EC"]
unexpected_ec = sum(x.get("condition") == "EC" for x in audit["missing_files"]) > 1
if intervals.empty or (audit["invalid_intervals"] or unexpected_missing or unexpected_ec or
                         audit["excluded_subjects_before_preprocessing"]) and not CFG["allow_recording_exclusions"]:
    raise RuntimeError("Review logs/dataset_audit.json for indexing errors and resolve them before continuing.")

## 4. CPU preprocessing and signal quality

Run a CPU smoke test with one recording per cohort, followed by incremental full preprocessing. Retain artifacts per Medley policy and store their recording-local window indices. Cache microvolt samples; Colab performs the single Z-score.

In [ ]:
def crop_and_window(raw, start, stop, line_freq):
    fs_orig = raw.info["sfreq"]
    if start > 0 or stop < raw.n_times / fs_orig:
        first = math.ceil(start * fs_orig - 1e-8)
        end = min(raw.n_times, math.ceil(stop * fs_orig - 1e-8))
        raw = raw.copy().crop(first / fs_orig, (end - 1) / fs_orig)
    else:
        raw = raw.copy()
    raw.set_eeg_reference("average", projection=False, verbose="ERROR")
    raw.filter(PP["l_freq"], min(PP["h_freq"], fs_orig / 2 - 1), verbose="ERROR")
    if line_freq < fs_orig / 2:
        raw.notch_filter(line_freq, verbose="ERROR")
    if abs(fs_orig - ENC["fs"]) > 1e-6:
        raw.resample(ENC["fs"], verbose="ERROR")
    x = raw.get_data() * 1e6
    length = round(ENC["fs"] * ENC["seg_sec"])
    n = x.shape[1] // length
    if not n:
        raise ValueError("No complete EEG windows in the selected interval.")
    windows = x[:, :n * length].reshape(19, n, length).transpose(1, 0, 2)
    flags = np.abs(windows).reshape(n, -1).max(1) > PP["flag_uv"]
    windows = np.ascontiguousarray(windows / ENC["amp_div"], dtype=np.float32)
    if windows.shape[1:] != (19, 400) or not np.isfinite(windows).all():
        raise ValueError("Invalid EEG cache output shape or non-finite signals.")
    return windows, flags

CACHE = PROJECT / "cache"
MANIFEST_PATH = CACHE / "preprocessing_manifest.json"
LINES = {name: cfg["line_freq"] for name, cfg in DATASET_CFG.items()}
REF_ALIASES = {name: cfg["ref_aliases"] for name, cfg in DATASET_CFG.items()}
CACHE_SPEC = dict(version=2, medley=MEDLEY_COMMIT, mne=versions["mne"], numpy=versions["numpy"],
                  scipy=versions["scipy"], preprocessing=PP, lines=LINES, ref_aliases=REF_ALIASES, channels=CHANNELS,
                  fs=ENC["fs"], seg_sec=ENC["seg_sec"], amp_div=ENC["amp_div"], montage=ENC["montage"])

def process_cache():
    index = pd.read_csv(INTERVAL_PATH, sep="\t", keep_default_na=False,
                        dtype={name: str for name in ("subject_uid", "person_uid", "source_subject_id", "session")})
    manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8")) if MANIFEST_PATH.exists() else {}
    entries = manifest.get("entries", {}) if manifest.get("spec") == CACHE_SPEC else {}
    rows, log, smoke_cohorts = [], [], set()
    groups = index.groupby("eeg_path", sort=True)
    estimated_bytes = int(((index.stop_sec - index.start_sec) / 2 + 1).sum()) * 19 * 400 * 4
    for count, (name, group) in enumerate(groups, 1):
        meta = group[["dataset", "subject_uid", "person_uid", "label_pd"]].drop_duplicates()
        if len(meta) != 1:
            raise ValueError(f"Multiple identities in recording {name}.")
        identity = meta.iloc[0].to_dict()
        c = identity["dataset"]
        if (not CFG["prepare_full_cache"] ) and c in smoke_cohorts:
            continue
        rec = dict(identity, eeg_path=name)
        try:
            path = Path(name)
            sidecars = sorted(p for p in path.parent.iterdir() if p.is_file() and
                              (p.name.startswith(path.stem.removesuffix("_eeg")) and p.suffix in
                               {".fdt", ".tsv", ".json"}))
            fingerprint = [(str(p), p.stat().st_size, p.stat().st_mtime_ns) for p in (path, *sidecars)]
            key = digest([CACHE_SPEC, fingerprint, group.to_dict("records")])
            target = CACHE / f"{key}.npy"
            entry = entries.get(key)
            valid = False
            if entry and target.is_file():
                try:
                    array = np.load(target, mmap_mode="r", allow_pickle=False)
                    valid = (array.shape == (entry["n_windows"], 19, 400) and array.dtype == np.float32
                             and sha_file(target) == entry["sha256"]
                             and isinstance(entry.get("flagged_offsets"), list))
                    del array
                except (OSError, ValueError, EOFError):
                    valid = False
            if not valid:
                tight_bytes = estimated_bytes if CFG["prepare_full_cache"] else int(
                    (((group.stop_sec - group.start_sec) / 2 + 1).sum()) * 19 * 400 * 4)
                if shutil.disk_usage(CACHE).free < tight_bytes + 128 * 2**20:
                    raise OSError("Insufficient disk space for EEG cache and temporary files.")
                # Same Medley channel handling as the baseline, before interval cropping.
                raw, interp = medley_eeg.load_raw(name, CHANNELS, REF_ALIASES[c], PP["min_good_channels"])
                parts, flag_parts = [], []
                with raw:
                    for part in group.itertuples(index=False):
                        windows, flags = crop_and_window(raw, part.start_sec, part.stop_sec, LINES[c])
                        parts.append(windows)
                        flag_parts.append(flags)
                array = np.concatenate(parts)
                def save_array(temporary):
                    with temporary.open("wb") as handle:
                        np.save(handle, array, allow_pickle=False)
                atomic(target, save_array)
                entry = dict(n_windows=len(array), flagged=sum(int(f.sum()) for f in flag_parts),
                             interpolated=interp, sha256=sha_file(target),
                             flagged_offsets=np.flatnonzero(np.concatenate(flag_parts)).tolist())
                entries[key] = entry
                write_json(MANIFEST_PATH, dict(spec=CACHE_SPEC, entries=entries))
            rows.append(dict(identity, cache_path=str(target), cache_key=key, **entry))
            rec.update(status="cached" if valid else "processed", **entry)
            smoke_cohorts.add(c)
        except Exception as exc:
            rec.update(status="failed", error=str(exc))
        log.append(rec)
        write_table(PROJECT / "logs/preprocessing_log.csv", pd.DataFrame(log), ",")
        print(f"{count}/{len(groups)} {c} {rec['status']}", flush=True)
    failures = [r for r in log if r["status"] == "failed"]
    audit["preprocessing_failures"] = failures
    audit["flagged_windows"] = sum(r["flagged"] for r in rows)
    audit["cache_recordings_usable"] = len(rows)
    audit["cache_recordings_attempted"] = len(log)
    audit["windows_by_subject"] = dict(Counter())
    for row in rows:
        uid = row["subject_uid"]
        audit["windows_by_subject"][uid] = audit["windows_by_subject"].get(uid, 0) + row["n_windows"]
    audit["cache_scope"] = "full" if CFG["prepare_full_cache"] else "smoke"
    if CFG["prepare_full_cache"]:
        audit["excluded_after_preprocessing"] = sorted(
            set(intervals.subject_uid) - {row["subject_uid"] for row in rows})
    write_json(PROJECT / "logs/dataset_audit.json", audit)
    if smoke_cohorts != set(ROOTS) or (failures and not CFG["allow_recording_exclusions"]):
        raise RuntimeError("Preprocessing incomplete. Review logs/preprocessing_log.csv.")
    return pd.DataFrame(rows)


# First exercise the actual channel loader and filtering on one recording per cohort.
full_cache_requested = CFG["prepare_full_cache"]
CFG["prepare_full_cache"] = False
try:
    smoke_cache = process_cache()
finally:
    CFG["prepare_full_cache"] = full_cache_requested
print("CPU smoke test windows", smoke_cache.groupby("dataset").n_windows.sum().to_dict())
cache_rows = process_cache() if full_cache_requested else None


## 5. Person-disjoint splits

Create the seed-0 70/15/15 split and summarize subject and window counts. Review unresolved identity links before training.

In [ ]:
def make_split(subjects, cfg):
    people = subjects.groupby("person_uid", sort=True).agg(
        dataset=("cohort", lambda x: "+".join(sorted(set(x)))), label_pd=("label_pd", "first")).reset_index()
    seed = cfg["seed"]
    holdout = cfg["val_fraction"] + cfg["test_fraction"]
    if min(cfg[k] for k in ("train_fraction", "val_fraction", "test_fraction")) <= 0 or not math.isclose(
            cfg["train_fraction"] + holdout, 1.0):
        raise ValueError("Split fractions must be positive and sum to one.")
    strata = people.dataset + "/" + people.label_pd.astype(str)
    # Use integer sizes to avoid floating point rounding of 121 × 0.30.
    holdout_n = len(people) - math.floor(len(people) * cfg["train_fraction"])
    test_n = math.ceil(holdout_n * cfg["test_fraction"] / holdout)
    for mode, labels in (("cohort+label", strata), ("label", people.label_pd)):
        try:
            train, rest = train_test_split(people, test_size=holdout_n, random_state=seed, stratify=labels)
            val, test = train_test_split(rest, test_size=test_n, random_state=seed, stratify=labels.loc[rest.index])
        except ValueError:
            continue
        parts = [train, val, test]
        if any(p.label_pd.nunique() != 2 for p in parts):
            continue
        if mode == "cohort+label" and any(labels.loc[p.index].nunique() != labels.nunique() for p in parts):
            continue
        people_split = pd.concat([part.assign(split=name)[["person_uid", "split"]]
                                 for name, part in zip(("train", "val", "test"), parts)])
        result = subjects.merge(people_split, on="person_uid", validate="many_to_one")
        if result.person_uid.nunique() != len(people) or result.subject_uid.duplicated().any():
            raise ValueError("Split membership is incomplete or duplicated.")
        return result, mode
    raise ValueError("Unable to form three person-disjoint splits with both classes.")

review = """# Participant overlap review

The supplied identity links govern the internal split. The external cohort names and
BIDS IDs alone provide no cross-study identity linkage. Pool identity linkage status:
{identity_links}.

Downstream PD-EEG MoCA (113 subjects): unresolved without a trusted cross-cohort
identity crosswalk or a documented review by the cohort data stewards. No downstream
records or outcomes are used here.

Pretraining: the pinned P-Base script lists PD-RS (the Singh et al. Iowa study) among
its inputs. The LEAD paper also reports BrainLat in its broader pretraining corpus.
Iowa and BrainLat are outside this fine-tuning pool. The P-Base script does not name
Turku, ds003490 or ds002778 as inputs. Cohort names and provenance are descriptive
evidence. Person-level crosswalks for the actual checkpoint are unavailable here.
Pretraining exposure and overlap with downstream evaluation are distinct questions.

Read the review and document the resolution or acknowledge the unresolved downstream
identity risk in CFG['overlap_acknowledgment'] before Task 7.

- https://arxiv.org/html/2502.01678v5
- https://github.com/DL4mHealth/LEAD/blob/dddee9c94f13b83d76f41ac9d452a74f84a76c5b/scripts/LEAD/pretrain/LEAD/P-Base.sh
"""
review = review.replace("{identity_links}", str(audit["identity_link_status"]))
atomic(PROJECT / "logs/overlap_review.md", lambda p: p.write_text(review, encoding="utf-8"))

finalized = None
if cache_rows is not None and CFG["prepare_full_cache"]:
    finalized, stratification = make_split(pool[pool.subject_uid.isin(cache_rows.subject_uid)], CFG)
    assert finalized.equals(make_split(pool[pool.subject_uid.isin(cache_rows.subject_uid)], CFG)[0])
    SPLIT_PATH = PROJECT / f"splits/pd_hc_subject_split_seed{CFG['seed']}_private.tsv"
    write_table(SPLIT_PATH, finalized)
    audit["split_stratification"] = stratification
    audit["split_distribution"] = finalized.groupby(["split", "cohort", "label_pd"]).size().rename(
        "subjects").reset_index().to_dict("records")
    write_json(PROJECT / "logs/dataset_audit.json", audit)

    assert finalized.groupby("person_uid").split.nunique().max() == 1
    assert finalized.groupby("subject_uid").split.nunique().max() == 1
    summary = cache_rows.merge(finalized[["subject_uid", "split"]], on="subject_uid", validate="many_to_one")
    audit["windows_by_split_and_cohort"] = summary.groupby(["split", "dataset"]).n_windows.sum().reset_index().to_dict("records")
    write_json(PROJECT / "logs/dataset_audit.json", audit)
    print(pd.DataFrame(audit["split_distribution"]))
    print(pd.DataFrame(audit["windows_by_split_and_cohort"]))


## 6. Build the local-only transfer ZIP

The ZIP contains lossless float32 [N, 400, 19] signals and random pseudonyms, splits, recording-relative window offsets, flag offsets, checksums and aggregate metadata. ds003490 and ds002778 declare CC0. The terms for cloud processing of the Turku recordings could not be verified from OSF. Keep the ZIP local until the dataset owner or steward and applicable institutional rules authorize the transfer. Record the authorization reference locally; the Colab notebook asks for confirmation before processing.

In [ ]:

def export_transfer():
    if finalized is None or cache_rows is None:
        raise RuntimeError("Complete CPU preprocessing and finalize the split first.")
    target = PROJECT / "transfer" / f"lead_pd_preprocessed_seed{CFG['seed']}.zip"
    if target.exists():
        raise FileExistsError(target)
    required = sum(Path(r.cache_path).stat().st_size for r in cache_rows.itertuples())
    if shutil.disk_usage(target.parent).free < required + 128 * 2**20:
        raise OSError("Insufficient disk for lossless ZIP and per-recording staging.")
    ids = {key: f"s-{uuid.uuid4().hex}" for key in finalized.subject_uid}
    people = {key: f"p-{uuid.uuid4().hex}" for key in finalized.person_uid.unique()}
    subjects = [dict(subject_uid=ids[r.subject_uid], person_uid=people[r.person_uid],
                     cohort=r.cohort, label_pd=int(r.label_pd), split=r.split)
                for r in finalized.itertuples(index=False)]
    recordings, checksums = [], {}
    # Subject/person mappings stay in memory. The ZIP contains only generated IDs.
    def write_bundle(temporary):
        with tempfile.TemporaryDirectory(dir=target.parent) as work, zipfile.ZipFile(
                temporary, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=3, allowZip64=True) as archive:
            staging = Path(work) / "recording.npy"
            for i, r in enumerate(cache_rows.itertuples(index=False)):
                if sha_file(r.cache_path) != r.sha256:
                    raise ValueError("Source cache changed after validation. Rerun preprocessing.")
                source = np.load(r.cache_path, mmap_mode="r", allow_pickle=False)
                if source.shape != (r.n_windows, 19, 400) or source.dtype != np.float32:
                    raise ValueError("Unexpected recording cache representation.")
                np.save(staging, np.ascontiguousarray(source.transpose(0, 2, 1), dtype="<f4"),
                        allow_pickle=False)
                del source
                name = f"windows/recording-{i:04d}.npy"
                archive.write(staging, name)
                checksums[name] = sha_file(staging)
                flags = sorted(set(r.flagged_offsets))
                if len(flags) != r.flagged or any(not 0 <= f < r.n_windows for f in flags):
                    raise ValueError("Invalid artifact flag offsets.")
                recordings.append(dict(subject_uid=ids[r.subject_uid], array=name,
                                       n_windows=r.n_windows, flagged_offsets=flags))
            # Window i in each array has recording-local index i. Sparse offsets mark flags.
            aggregate = {
                "eligible_counts": audit["eligible_counts"],
                "identity_link_status": audit["identity_link_status"],
                "selected_subjects": len(subjects),
                "selected_recordings": len(recordings),
                "signal_excluded_subject_count": len(set(pool.subject_uid) - set(finalized.subject_uid)),
                "flagged_windows": audit["flagged_windows"],
                "split_distribution": audit["split_distribution"],
                "windows_by_split_and_cohort": audit["windows_by_split_and_cohort"],
            }
            cfg = {k: v for k, v in CFG.items() if k not in {
                "overlap_acknowledgment", "turku_transfer_permission_reference",
                "prepare_full_cache", "cpu_smoke", "allow_recording_exclusions", "unm_marker_guard_sec"}}
            manifest = dict(format_version=1, layout="N,T,C", dtype="float32", normalized=False,
                            normalization="Medley window/channel z-score in Colab",
                            medley=MEDLEY_COMMIT, lead=LEAD_COMMIT,
                            preprocessing=CACHE_SPEC, training=cfg, audit=aggregate,
                            subjects=subjects, recordings=recordings, sha256=checksums)
            data = json.dumps(manifest, indent=2, allow_nan=False).encode()
            archive.writestr("manifest.json", data)
            archive.writestr("manifest.sha256", hashlib.sha256(data).hexdigest())
    atomic(target, write_bundle)
    with zipfile.ZipFile(target) as archive:
        if archive.testzip() is not None:
            raise ValueError("ZIP integrity validation failed.")
        uncompressed = sum(info.file_size for info in archive.infolist())
    summary = dict(path=str(target), compressed_bytes=target.stat().st_size,
                   uncompressed_bytes=uncompressed, sha256=sha_file(target))
    write_json(PROJECT / "logs/transfer_size.json", summary)
    print(summary)
    if not CFG["turku_transfer_permission_reference"].strip():
        print("Keep this archive local. Turku cloud transfer permission has not been verified.")

export_transfer()
